## Medical ANN Classifier with Zero-Knowledge Machine Learning (zkML)

## Objective
The objective of this project is to develop an Artificial Neural Network (ANN) classifier using a medical dataset and integrate it with Zero-Knowledge Machine Learning (zkML). The trained model should generate predictions for unseen medical records and also produce a cryptographic proof verifying that the prediction was computed by the trained model. This enhances trust, privacy, and security in AI-based medical diagnosis.

## Installing Required Libraries

This project requires several Python libraries for implementing Artificial Neural Networks (ANN) and Zero-Knowledge Machine Learning (zkML).

- **PyTorch** is used to build and train the ANN model.
- **Pandas** is used for loading and handling the medical dataset.
- **Scikit-learn** is used for data preprocessing.
- **ONNX** is used to export the trained neural network into a standard format.
- **EZKL** is the zkML framework used to generate and verify Zero-Knowledge Proofs.

The required packages are installed before starting the implementation.

In [ ]:
!pip install ezkl onnx torch scikit-learn pandas -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 47.0 MB/s eta 0:00:00


In [ ]:
!pip install nest_asyncio -q
import nest_asyncio
nest_asyncio.apply()

## Loading the Medical Dataset

The Pima Indians Diabetes Dataset is used in this project. It is a widely used benchmark dataset for binary classification problems in healthcare.

Each record represents one patient and contains eight medical features:

- Pregnancies
- Glucose
- Blood Pressure
- Skin Thickness
- Insulin
- Body Mass Index (BMI)
- Diabetes Pedigree Function
- Age

The target variable is **Outcome**, where:

- **0** represents a non-diabetic patient.
- **1** represents a diabetic patient.

The dataset is loaded into a Pandas DataFrame for preprocessing and model training.

In [ ]:
import os
os.environ["RUST_LOG"] = "trace"
import ezkl, asyncio

In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin","BMI","DPF","Age","Outcome"]
df = pd.read_csv(url, names=cols)
df.head()


## Artificial Neural Network Design and Training

In this section, the complete Artificial Neural Network (ANN) pipeline is implemented. First, the medical dataset is divided into training and testing sets. Since the medical attributes have different numerical ranges, feature scaling is performed using StandardScaler to normalize the input values.

An Artificial Neural Network is then designed using the PyTorch framework. The network consists of an input layer containing eight medical features, two hidden layers with ReLU activation functions, and one output layer with a Sigmoid activation function for binary classification.

The model is trained using the Binary Cross Entropy (BCE) loss function and the Adam optimizer. During each epoch, forward propagation computes the predictions, the loss is calculated, backpropagation updates the weights, and the optimizer minimizes the prediction error. After completing all training epochs, the final training loss is displayed, indicating how well the network has learned the relationship between the medical features and diabetes prediction.

In [ ]:
import torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

X = df.drop("Outcome", axis=1).values.astype("float32")
y = df["Outcome"].values.astype("float32")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)

class ANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

model = ANN()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCELoss()
Xt = torch.tensor(X_train); yt = torch.tensor(y_train).unsqueeze(1)

for epoch in range(150):
    opt.zero_grad()
    out = model(Xt)
    loss = loss_fn(out, yt)
    loss.backward()
    opt.step()

print("final loss:", loss.item())


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import torch

model.eval()

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

with torch.no_grad():
    outputs = model(X_test_tensor)
    predictions = (outputs >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print("="*50)
print("Model Evaluation")
print("="*50)
print(f"Test Accuracy : {accuracy*100:.2f}%")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, predictions))

print("\nClassification Report")
print(classification_report(y_test, predictions))

## Trained ANN Model

After training, the neural network has learned the mapping between the patient's medical parameters and the corresponding diabetes outcome. The trained weights now represent the knowledge acquired from the dataset and will be used for future predictions.

The trained model serves as the machine learning component that will later be converted into the ONNX format for Zero-Knowledge Machine Learning (zkML) implementation.

## Exporting the Model to ONNX

Zero-Knowledge Machine Learning frameworks such as EZKL cannot directly process PyTorch models. Therefore, the trained ANN must first be converted into the Open Neural Network Exchange (ONNX) format.

ONNX provides a standardized representation of the neural network architecture, model parameters, and computational graph. This exported model acts as the bridge between the trained ANN and the zkML framework.

In [ ]:
!pip install onnxscript -q


In [ ]:
dummy_input = torch.tensor(X_train[:1])
torch.onnx.export(model, dummy_input, "network.onnx",
                   input_names=["input"], output_names=["output"],
                  opset_version=13)


## ONNX Model Successfully Generated

The trained ANN has now been exported into the ONNX format. This file contains the complete computational graph of the neural network along with all learned parameters.

This ONNX model will be used by EZKL to construct a Zero-Knowledge circuit capable of generating and verifying cryptographic proofs for neural network inference.

## Preparing Input for Zero-Knowledge Proof Generation

Before generating a Zero-Knowledge Proof, a sample input must be prepared. A patient record from the testing dataset is converted into JSON format because EZKL accepts inference inputs in JSON representation.

This JSON file contains the feature values that will be supplied to the neural network during proof generation.

In [ ]:
import json
sample = X_test[0:1].tolist()
data = dict(input_data=sample)
json.dump(data, open("input.json", "w"))

In [ ]:
import json
print(json.load(open("input.json")))


## Verifying the Exported ONNX Model

The exported ONNX model is validated to ensure that it has been generated correctly. The computational graph is printed to verify the structure of the neural network and to confirm that all layers, weights, and activation functions have been successfully exported.

This verification step helps detect any incompatibility before the model is passed to the zkML framework.

In [ ]:
import onnx
m = onnx.load("network.onnx")
onnx.checker.check_model(m)
print(m.opset_import)
print(onnx.helper.printable_graph(m.graph))


In [ ]:
import onnx
m = onnx.load("network.onnx")
print("ir_version:", m.ir_version)
print("opset:", m.opset_import)


## Adjusting ONNX Compatibility

Different versions of EZKL support specific ONNX Intermediate Representation (IR) versions. Therefore, the IR version of the exported ONNX model is modified to ensure compatibility with the installed EZKL framework.

This compatibility adjustment allows successful compilation of the neural network into a Zero-Knowledge circuit.

In [ ]:
m.ir_version = 8
onnx.save(m, "network.onnx")
print("done, new ir_version:", m.ir_version)


## Zero-Knowledge Machine Learning (zkML) Pipeline

After training the Artificial Neural Network (ANN) and exporting it to the ONNX format, the model is integrated with the **EZKL** framework to enable **Zero-Knowledge Machine Learning (zkML)**.

The zkML pipeline performs the following steps:

- Generate the configuration settings.
- Calibrate numerical precision for the model.
- Compile the ONNX model into a Zero-Knowledge circuit.
- Generate the proving and verification keys.
- Create a witness for the given input.
- Generate a Zero-Knowledge Proof.
- Verify the generated proof.

This process ensures that the model's prediction is **cryptographically verifiable** while keeping both the model parameters and sensitive medical data private.

In [ ]:
import ezkl, asyncio

ezkl.gen_settings("network.onnx", "settings.json")
ezkl.calibrate_settings("input.json", "network.onnx", "settings.json", "resources")
ezkl.compile_circuit("network.onnx", "network.compiled", "settings.json")
asyncio.run(ezkl.get_srs("settings.json"))
ezkl.setup("network.compiled", "vk.key", "pk.key")
ezkl.gen_witness("input.json", "network.compiled", "witness.json")
ezkl.prove("witness.json", "network.compiled", "pk.key", "proof.json")
result = ezkl.verify("proof.json", "settings.json", "vk.key")
print("Proof valid:", result)


## Zero-Knowledge Proof Verification

Successful verification confirms that the neural network inference was executed exactly as defined by the trained model. The generated cryptographic proof enables any verifier to validate the computation without re-running the neural network and without revealing confidential information.

This demonstrates the fundamental objective of Zero-Knowledge Machine Learning: providing trust, transparency, and privacy in machine learning inference.

## Performing Secure Inference on New Data

One of the primary objectives of this project is to allow the trained ANN to process previously unseen medical records. For every new input, the neural network generates a prediction, followed by the creation of a Zero-Knowledge Proof.

This proof confirms that the prediction was genuinely produced by the trained model and that the computation was performed correctly, without exposing either the model parameters or the patient's private medical information.

In [ ]:
def predict_and_prove(new_input):
    import json
    json.dump({"input_data": new_input}, open("input.json", "w"))
    ezkl.gen_witness("input.json", "network.compiled", "witness.json")
    ezkl.prove("witness.json", "network.compiled", "pk.key", "proof.json")
    valid = ezkl.verify("proof.json", "settings.json", "vk.key")
    print("Proof valid:", valid)

predict_and_prove(X_test[3:4].tolist())


## Repeated Proof Verification

To evaluate the reliability of the zkML pipeline, the proof-generation and verification process is tested on multiple unseen test samples. Each sample is passed through the trained model and the corresponding zero-knowledge proof is generated and verified. The verification result is recorded for every sample.

In [ ]:
# Test proof verification on multiple unseen samples

num_samples = 10
successful = 0

for i in range(num_samples):
    new_input = X_test[i:i+1].tolist()

    json.dump({"input_data": new_input}, open("input.json", "w"))

    ezkl.gen_witness(
        "input.json",
        "network.compiled",
        "witness.json"
    )

    ezkl.prove(
        "witness.json",
        "network.compiled",
        "pk.key",
        "proof.json"
    )

    valid = ezkl.verify(
        "proof.json",
        "settings.json",
        "vk.key"
    )

    print(f"Sample {i+1}: Proof valid = {valid}")

    if valid:
        successful += 1

print("\nProof Verification Summary")
print(f"Successful proofs: {successful}/{num_samples}")
print(f"Verification success rate: {(successful/num_samples)*100:.2f}%")

## Conclusion

This project successfully demonstrates the integration of Artificial Neural Networks with Zero-Knowledge Machine Learning for secure medical diagnosis. A diabetes prediction model was trained using a medical dataset, exported to the ONNX format, and integrated with EZKL to generate verifiable proofs of inference.

By combining machine learning with cryptographic verification, the proposed approach enhances trust, security, and privacy while ensuring that neural network predictions can be independently verified without revealing confidential information.